# Практическая работа №3: Работа с реальными данными
**Вариант 4**

Цель: научиться читать файлы разных форматов (JSON Lines, CSV), объединять несколько источников в один DataFrame и сохранять результат в разных форматах.

Задания варианта:
1. Считать файл с событиями в формате JSON Lines.
2. Посчитать количество событий на каждого пользователя.
3. Объединить результат с таблицей заказов, добавив к каждому пользователю количество событий.
4. Найти пользователей, у которых есть заказы, но нет событий.
5. Сохранить объединённую таблицу в `parquet` и `csv`, сравнить размер файлов.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Задание 1. Чтение файла событий в формате JSON Lines

In [2]:
df_events = pd.read_json("data/events_lines.json", lines=True)

print(df_events.head())
df_events.info()

   event_id  user_id          event_time   event_type         page
0      2001     1029 2024-03-19 00:34:38     purchase  /product/20
1      2002     1056 2024-03-25 17:16:08  add_to_cart        /cart
2      2003     1081 2024-02-06 06:09:49  add_to_cart  /product/30
3      2004     1005 2024-03-20 04:35:23     purchase   /product/5
4      2005     1033 2024-02-21 09:51:53        click  /product/11
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    2000 non-null   int64         
 1   user_id     2000 non-null   int64         
 2   event_time  2000 non-null   datetime64[us]
 3   event_type  2000 non-null   str           
 4   page        2000 non-null   str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 111.0 KB


## Задание 2. Количество событий на каждого пользователя

In [3]:
events_per_user = df_events.groupby("user_id").size().reset_index(name="events_count")

print(events_per_user.head(10))

   user_id  events_count
0     1001            29
1     1002            25
2     1003            21
3     1004            22
4     1005            21
5     1006            15
6     1007            18
7     1008            22
8     1009            19
9     1010            15


## Задание 3. Объединение с таблицей заказов

In [4]:
orders_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders = pd.concat([pd.read_csv(f) for f in orders_files], ignore_index=True)

df_orders_events = df_orders.merge(events_per_user, on="user_id", how="left")
df_orders_events["events_count"] = df_orders_events["events_count"].fillna(0).astype("int64")

print(df_orders_events.head(10))

   order_id  user_id       order_datetime  product_id     price  quantity  \
0         1     1044  2024-01-03 18:39:26          26  14038.15         1   
1         2     1079  2024-01-03 17:45:43          16   9172.60         1   
2         3     1041  2024-01-06 22:46:38          25   9035.26         2   
3         4     1083  2024-01-08 21:03:51           9  15234.33         3   
4         5     1068  2024-01-03 23:26:53          24   4134.38         2   
5         6     1069  2024-01-15 13:09:44          28  19359.94         3   
6         7     1080  2024-01-29 08:04:28           6   2859.45         3   
7         8     1044  2024-01-15 13:40:56           5  14095.22         4   
8         9     1085  2024-01-24 19:26:48          12   5980.06         1   
9        10     1001  2024-01-22 19:11:48          24  13397.56         4   

     status  events_count  
0  refunded            20  
1      paid            16  
2      paid            21  
3      paid            19  
4      paid 

## Задание 4. Пользователи с заказами, но без событий

In [5]:
users_without_events = set(df_orders["user_id"]) - set(df_events["user_id"])

print("Пользователей с заказами, но без событий:", len(users_without_events))
print(sorted(users_without_events))

Пользователей с заказами, но без событий: 0
[]


## Задание 5. Сохранение в parquet и csv, сравнение размеров

In [6]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

csv_path = results_dir / "orders_with_events.csv"
parquet_path = results_dir / "orders_with_events.parquet"

df_orders_events.to_csv(csv_path, index=False)
df_orders_events.to_parquet(parquet_path, index=False)

csv_size = csv_path.stat().st_size / 1024
parquet_size = parquet_path.stat().st_size / 1024

print(f"CSV: {csv_size:.1f} KB")
print(f"Parquet: {parquet_size:.1f} KB")
print(f"Parquet меньше CSV в {csv_size / parquet_size:.1f} раза")

CSV: 766.1 KB
Parquet: 389.7 KB
Parquet меньше CSV в 2.0 раза
